In [1]:
import torch
import pytorch_lightning as pl
from transformers import WhisperProcessor, WhisperForConditionalGeneration, get_linear_schedule_with_warmup
from datasets import load_dataset, Audio
from torch.utils.data import DataLoader
from pytorch_lightning import Trainer
from dataclasses import dataclass
from typing import Any, Dict, List, Union


/home/yopsa_matb/audio_lab3/myenv/lib/python3.14/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
/home/yopsa_matb/audio_lab3/myenv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
print("=== 1. ІНІЦІАЛІЗАЦІЯ (WHISPER-SMALL) ===")
model_id = "openai/whisper-base"
processor = WhisperProcessor.from_pretrained(model_id)
processor.tokenizer.set_prefix_tokens(language="uk", task="transcribe") 


=== 1. ІНІЦІАЛІЗАЦІЯ (WHISPER-SMALL) ===


In [ ]:
print("=== 2. ПІДГОТОВКА ДАНИХ (ПРЯМИЙ ПЕРЕКЛАД) ===")
dataset_uk_full = load_dataset("google/fleurs", "uk_ua", split="train")
uk_text_map = {item['id']: item['transcription'] for item in dataset_uk_full}

dataset_en_full = load_dataset("google/fleurs", "en_us", split="train")
dataset_aligned = dataset_en_full.filter(lambda x: x["id"] in uk_text_map)

dataset_small = dataset_aligned.select(range(2500))
dataset_small = dataset_small.cast_column("audio", Audio(sampling_rate=16000))

def prepare_ast_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    
    uk_translation = uk_text_map[batch["id"]] 
    batch["labels"] = processor.tokenizer(uk_translation).input_ids
    
    batch["src_text"] = batch["transcription"]
    batch["ref_text"] = uk_translation         
    return batch

prepared_fleurs = dataset_small.map(prepare_ast_dataset, remove_columns=dataset_small.column_names, num_proc=4)


=== 2. ПІДГОТОВКА ДАНИХ (ПРЯМИЙ ПЕРЕКЛАД) ===


Map (num_proc=4): 100%|██████████| 2500/2500 [01:16<00:00, 32.68 examples/s]


In [ ]:
print("=== 3. МОДЕЛЬ ТА НАВЧАННЯ ===")
from peft import LoraConfig, get_peft_model


@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features):
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch
    
class ASTLightningModule(pl.LightningModule):
    def __init__(self, learning_rate=1e-4): # Для LoRA потрібен більший LR
        super().__init__()
        
        base_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
        
        config = LoraConfig(
            r=32,
            lora_alpha=64,
            target_modules=["q_proj", "v_proj"],
            lora_dropout=0.05,
            bias="none"
        )
        
        # 3. Обгортаємо модель
        self.model = get_peft_model(base_model, config)
        self.model.print_trainable_parameters() 
        
        self.learning_rate = learning_rate

    def forward(self, input_features, labels):
        return self.model(input_features=input_features, labels=labels)

    def training_step(self, batch, batch_idx):
        outputs = self(batch["input_features"], batch["labels"])
        loss = outputs.loss
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        outputs = self(batch["input_features"], batch["labels"])
        loss = outputs.loss
        self.log("val_loss", loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.learning_rate)
        scheduler = get_linear_schedule_with_warmup(
            optimizer, num_warmup_steps=50, num_training_steps=self.trainer.estimated_stepping_batches
        )
        return [optimizer], [{"scheduler": scheduler, "interval": "step"}]

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
train_dataloader = DataLoader(prepared_fleurs, batch_size=2, collate_fn=data_collator, shuffle=True)
val_dataloader = DataLoader(prepared_fleurs, batch_size=2, collate_fn=data_collator)

model = ASTLightningModule()

trainer = Trainer(
    max_epochs=5, 
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    gradient_clip_val=1.0,
    log_every_n_steps=5,
    enable_progress_bar=True
)

trainer.fit(model, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)


=== 3. МОДЕЛЬ ТА НАВЧАННЯ ===


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/home/yopsa_matb/audio_lab3/myenv/lib/python3.14/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint,

trainable params: 3,538,944 || all params: 245,273,856 || trainable%: 1.4429


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.
/home/yopsa_matb/audio_lab3/myenv/lib/python3.14/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/yopsa_matb/audio_lab3/myenv/lib/python3.14/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


┏━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ PeftModel │  245 M │ train │     0 │
└───┴───────┴───────────┴────────┴───────┴───────┘

...

/home/yopsa_matb/audio_lab3/myenv/lib/python3.14/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" 
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/home/yopsa_matb/audio_lab3/myenv/lib/python3.14/site-packages/pytorch_lightning/trainer/connectors/data_connector.
py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of
the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.

You're using a WhisperTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


/home/yopsa_matb/audio_lab3/myenv/lib/python3.14/site-packages/pytorch_lightning/loops/fit_loop.py:534: Found 350 
module(s) in eval mode at the start of training. This may lead to unexpected behavior during training. If this is 
intentional, you can ignore this warning.

`Trainer.fit` stopped: `max_epochs=5` reached.


In [ ]:

print("=== 4. ГЕНЕРАЦІЯ ТА COMET МЕТРИКА ===")
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

src_texts, ref_texts, hyp_texts = [], [], []

with torch.no_grad():
    for i, batch in enumerate(val_dataloader):
        if i >= 5:
            break
            
        input_features = batch["input_features"].to(device)
        
        predicted_ids = model.model.generate(
            input_features, 
            language="uk", 
            task="transcribe",
            num_beams=5,
            repetition_penalty=1.5,
            no_repeat_ngram_size=3,
            max_new_tokens=100
        )
        
        transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)
        
        labels = batch["labels"].cpu()
        labels[labels == -100] = processor.tokenizer.pad_token_id
        references = processor.batch_decode(labels, skip_special_tokens=True)
        
        for hyp, ref in zip(transcription, references):
            print(f"Генерація (Укр): {hyp}")
            print(f"Еталон (Укр):    {ref}")
            print("-" * 30)
            hyp_texts.append(hyp)
            ref_texts.append(ref)
            src_texts.append("Placeholder english text for COMET to run successfully")



=== 4. ГЕНЕРАЦІЯ ТА COMET МЕТРИКА ===


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Генерація (Укр): турнадо є спінком колеминою дуже низькою пресією водою який збережить повітря на воді вибору та на подорожі
Еталон (Укр):    торнадо це стовп повітря із дуже низьким тиском котрий обертаючись засмоктує навколишнє повітря всередину та вгору
------------------------------
Генерація (Укр): йогурт-спікор власу ньют гінгріч прийшов у сьогодні з 32%
Еталон (Укр):    колишній спікер палати представників сша ньют гінгріч зайняв друге місце отримавши 32 відсотки
------------------------------
Генерація (Укр): ця нерава можна дозволювати так швидко відтягнути тим що допомагає збережити тим ніж потенційної триви
Еталон (Укр):    ці нервові імпульси проходять тілом дуже швидко що допомагає захищати його від будь-яких потенційних загроз
------------------------------
Генерація (Укр): після 24 вересня 1759 артур гіннес признався 9000-річним лісом на речі сент-жеймської дворі в дубліній ірландії
Еталон (Укр):    24 вересня 1759 артур гіннесс підписав контракт на 9000-річну аренду пив

In [ ]:
from comet import download_model, load_from_checkpoint
import torch

print("Завантаження моделі COMET...")
model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

data = [{"src": src, "mt": hyp, "ref": ref} for src, ref, hyp in zip(src_texts, ref_texts, hyp_texts)]

print("Розрахунок метрики COMET...")
model_output = comet_model.predict(data, batch_size=8, gpus=1 if torch.cuda.is_available() else 0, num_workers=0)

print("-" * 30)
print(f"ФІНАЛЬНИЙ COMET SCORE: {model_output.system_score:.4f}")
print("-" * 30)

Завантаження моделі COMET...


Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 38269.20it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.1. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.
/home/yopsa_matb/audio_lab3/myenv/lib/python3.14/site-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try insta

Розрахунок метрики COMET...


/home/yopsa_matb/audio_lab3/myenv/lib/python3.14/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Predicting DataLoader 0: 100%|██████████| 2/2 [00:00<00:00,  2.53it/s]


------------------------------
ФІНАЛЬНИЙ COMET SCORE: 0.4690
------------------------------
